# 🏥 Clínica de estética — Análise de Dados

Fluxo completo de análise dos dados da clínica:

- **Parte 1 — Carregamento e inspeção**
- **Parte 2 — Diagnóstico de qualidade**
- **Parte 3 — Limpeza**
- **Parte 4 — EDA (Análise Exploratória)**
- **Parte 5 — KPIs**
- **Parte 6 — Tendência de faturamento**

## Setup

In [122]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import gspread
from pathlib import Path

In [123]:
JSON_KEY = Path("skillsgo-501021-2a0589249e00.json")
FOLDER_ID = "1PETukXVP5tb0CaHQ1JF-XwiCHmRKq5ol"

gc = gspread.service_account(filename=JSON_KEY)

In [124]:
JSON_KEY = Path("skillsgo-501021-2a0589249e00.json")  # Caminho da chave da Service Account (gerada no GCP)
FOLDER_ID = "1PETukXVP5tb0CaHQ1JF-XwiCHmRKq5ol"          # ID da pasta do Google Drive onde as planilhas estão

# Autentica com o Google usando a Service Account, retorna um "cliente" pra interagir com Sheets/Drive
gc = gspread.service_account(filename=JSON_KEY)  

> **PROMPT**

> Preciso de duas funções em Python usando a lib gspread: uma pra ler uma planilha do Google Sheets (por nome, dentro de uma pasta específica > do Drive) e transformar em DataFrame do pandas, e outra pra fazer o caminho inverso — salvar um DataFrame de volta numa planilha existente, > sobrescrevendo o conteúdo. A autenticação já está pronta via Service Account.

In [125]:
def ler_planilha(nome):
    # Abre a planilha pelo nome dentro da pasta específica (FOLDER_ID),
    # pega a primeira aba (sheet1), lê todas as linhas como dicionário,
    # e transforma isso num DataFrame do pandas
    return pd.DataFrame(gc.open(nome, folder_id=FOLDER_ID).sheet1.get_all_records())


def salvar_planilha(df, nome):
    export = df.copy()  # Cria uma cópia do DataFrame pra não alterar o original ao exportar

    # O Google Sheets não entende tipo "datetime" do pandas — precisa virar texto antes de enviar
    for col in export.columns:
        if pd.api.types.is_datetime64_any_dtype(export[col]):
            export[col] = export[col].dt.strftime("%Y-%m-%d")  # Formata data como texto YYYY-MM-DD

    # Monta a lista de dados no formato que o Sheets espera:
    # primeira linha = cabeçalho (nomes das colunas), demais linhas = valores como texto
    dados = [export.columns.tolist()] + export.astype(str).values.tolist()

    try:
        sh = gc.open(nome, folder_id=FOLDER_ID)  # Tenta abrir a planilha de destino pelo nome
    except gspread.SpreadsheetNotFound:
        # Se a planilha não existir ainda, avisa com instrução clara de como criar/compartilhar
        raise RuntimeError(
            f"Planilha '{nome}' não encontrada em FOLDER_ID={FOLDER_ID}. "
            f"Crie-a manualmente no Drive e compartilhe com "
            f"skillsgo@skillsgo-501021.iam.gserviceaccount.com como Editor."
        )

    sh.sheet1.clear()          # Limpa todo o conteúdo atual da aba antes de escrever
    sh.sheet1.update(dados)    # Escreve os novos dados (cabeçalho + linhas) na planilha
    print(f"✓ {nome}")         # Confirma no console que a planilha foi atualizada com sucesso

In [126]:
planilhas = gc.list_spreadsheet_files()
for p in planilhas:
    print(p["name"])

financeiro
financeiro_limpo
estoque_limpo
agenda_limpa
agenda
clientes_limpo
estoque
clientes


In [127]:
df_agenda = ler_planilha("agenda")
df_clientes = ler_planilha("clientes")
df_estoque = ler_planilha("estoque")
df_financeiro = ler_planilha("financeiro")

print(f"agenda     → {df_agenda.shape[0]:,} linhas x {df_agenda.shape[1]} colunas")
print(f"clientes   → {df_clientes.shape[0]:,} linhas x {df_clientes.shape[1]} colunas")
print(f"estoque    → {df_estoque.shape[0]:,} linhas x {df_estoque.shape[1]} colunas")
print(f"financeiro → {df_financeiro.shape[0]:,} linhas x {df_financeiro.shape[1]} colunas")


agenda     → 1,222 linhas x 7 colunas
clientes   → 200 linhas x 10 colunas
estoque    → 15 linhas x 5 colunas
financeiro → 758 linhas x 5 colunas


## Inspeção inicial

Antes de limpar, precisamos entender o que temos.

### Agenda

In [128]:
print(df_agenda.dtypes)
df_agenda.head(5)

data               object
horario            object
cliente_id         object
profissional       object
procedimento       object
forma_pagamento    object
status             object
dtype: object


,data,horario,cliente_id,profissional,procedimento,forma_pagamento,status
0,2026-01-01,09:00,CLI0060,Biom. Juliana Faria,Microagulhamento,Particular,realizado
1,2026-01-01,09:00,CLI0073,Biom. Juliana Faria,Peeling químico,Particular,realizado
2,2026-01-01,09:00,CLI0123,Dra. Camila Torres,Depilação a laser,Pix,cancelado
3,2026-01-01,09:30,CLI0093,Est. Renata Souza,Botox,Cartão crédito,confirmado
4,2026-01-01,10:30,CLI0033,Dra. Camila Torres,Botox,Convênio empresa,confirmado


Todas as colunas estão como object — isso significa texto. A coluna data precisa ser convertida para datetime e horario pode ser tratado como string mesmo. Vamos corrigir os tipos na limpeza.

In [129]:
# Valores únicos nas colunas categóricas
print("Status únicos:")
print(df_agenda['status'].value_counts(dropna=False))


Status únicos:
status
realizado     723
cancelado     253
confirmado    246
Name: count, dtype: int64


In [130]:
print("Procedimentos únicos:")
print(df_agenda['procedimento'].value_counts(dropna=False))

Procedimentos únicos:
procedimento
Harmonização facial     151
Microagulhamento        145
Preenchimento facial    143
Preenchimento labial    142
Avaliação inicial       141
Botox                   131
Limpeza de pele         125
Peeling químico         124
Depilação a laser       120
Name: count, dtype: int64


In [131]:
print("Pagamentos únicos:")
print(df_agenda['forma_pagamento'].value_counts(dropna=False))

Pagamentos únicos:
forma_pagamento
Pix                 277
Particular          270
Convênio empresa    252
Cartão crédito      219
Cartão débito       204
Name: count, dtype: int64


### Estoque

In [132]:
print(df_estoque.dtypes)
df_estoque

item                 object
categoria            object
quantidade_atual      int64
quantidade_minima     int64
ultima_reposicao     object
dtype: object


,item,categoria,quantidade_atual,quantidade_minima,ultima_reposicao
0,Toxina botulínica (unid),Insumo médico,-3,10,2026-06-15
1,Ácido hialurônico 1ml,Insumo médico,15,10,2026-05-27
2,Luva descartável (cx 100),EPI,4,8,2026-05-29
3,Agulha de preenchimento (cx 50),Insumo médico,22,15,2026-06-06
4,Sérum vitamina C 30ml,Cosmético,18,10,2026-05-03
5,Protetor solar FPS50 (unid),Cosmético,3,-1,2026-06-03
6,Peeling enzimático 100g,Cosmético,12,8,2026-05-15
7,Máscara facial hidratante (cx),Cosmético,7,10,2026-05-14
8,Álcool 70% 1L,Higiene,25,10,2026-05-20
9,Papel toalha (fardo),Higiene,6,5,2026-05-23


Os tipos fazem sentido: `quantidade_atual` e `quantidade_minima` como `int64`, e `ultima_reposicao` como `object` porque ainda é texto, vai virar `datetime` na limpeza.
- `Toxina botulínica` com `-3` e `Protetor solar` com mínimo `-1`. Corrigir na limpeza.

### Financeiro

In [133]:
print(df_financeiro.dtypes)
df_financeiro.head(5)

data         object
tipo         object
categoria    object
valor        object
descricao    object
dtype: object


,data,tipo,categoria,valor,descricao
0,2026-01-01,entrada,Depilação a laser,1415.87,Pagamento - Olívia Pimenta
1,2026-01-01,entrada,Harmonização facial,1823.2,Pagamento - Jade Carvalho
2,2026-01-01,entrada,Harmonização facial,730.67,Pagamento - Srta. Maria Alice Guerra
3,2026-01-01,entrada,Pacote tratamento,1422.81,Pagamento - Alexia Viana
4,2026-01-01,saida,Cosméticos,4628.94,Pagamento - Insumos médicos


Todas as colunas como object — incluindo valor, que deveria ser numérico. Isso acontece porque alguns valores têm R$ e vírgula como decimal, o que impediu o pandas de inferir o tipo automaticamente. Na limpeza vamos converter com uma função específica para o formato monetário brasileiro. 

In [134]:
print("Tipos únicos:")
print(df_financeiro['tipo'].value_counts(dropna=False))

Tipos únicos:
tipo
entrada    693
saida       65
Name: count, dtype: int64


In [135]:
df_financeiro['categoria'].unique()

array(['Depilação a laser', 'Harmonização facial', 'Pacote tratamento',
       'Cosméticos', 'Aluguel', 'Peeling', 'Microagulhamento', 'Botox',
       'Preenchimento', 'Folha de pagamento', 'Marketing digital',
       'Limpeza de pele', 'Avaliação', 'Impostos', 'Equipamentos',
       'Insumos médicos', 'Avaliação inicial', 'Preenchimento facial',
       'Peeling químico', 'Preenchimento labial'], dtype=object)

> **PROMPT**

> No DataFrame df_financeiro, verifica se a coluna valor tem valores que ainda estão como texto com 'R$' (ao invés de número). Mostra quantos são e imprime as 5 primeiras linhas com esse problema, com as colunas data, tipo e valor.

In [136]:
# Valores com R$ e vírgula — colagem de planilha financeira
tem_reais = df_financeiro['valor'].astype(str).str.contains(r'R\$', regex=True)
print(f"Valores com R$: {tem_reais.sum()}")
print(df_financeiro[tem_reais][['data', 'tipo', 'valor']].head(5))

Valores com R$: 5
           data     tipo        valor
230  2026-02-24  entrada  R$ 1.977,29
296  2026-03-10    saida  R$ 4.720,86
399  2026-04-03  entrada    R$ 640,13
602  2026-05-22  entrada  R$ 1.483,26
664  2026-06-05  entrada  R$ 1.607,67


> **PROMPT**

> No DataFrame df_financeiro, converte a coluna valor pra número e verifica se existe algum valor negativo — o que não deveria acontecer, já que entradas de caixa não podem ser negativas. Mostra quantos casos tem e imprime uma amostra com data, tipo e valor. 

In [137]:
# Converte para float, tratando erros como NaN
df_financeiro['valor'] = pd.to_numeric(df_financeiro['valor'], errors='coerce')

# Procura valores negativos - normalmente não deveria haver saídas negativas
valores_negativos = df_financeiro[df_financeiro['valor'] < 0]

print(f"Quantidade de registros com valor negativo: {len(valores_negativos)}")
print(valores_negativos[['data', 'tipo', 'valor']].head(5))

Quantidade de registros com valor negativo: 5
           data     tipo    valor
259  2026-03-03  entrada  -913.98
355  2026-03-24  entrada  -952.17
653  2026-06-03  entrada -1327.29
746  2026-06-24    saida  -980.00
753  2026-06-30    saida -3200.00


### Nulos e duplicados

In [138]:
# Nulos — Agenda
df_agenda.isnull().sum()

data               0
horario            0
cliente_id         0
profissional       0
procedimento       0
forma_pagamento    0
status             0
dtype: int64

In [139]:
# Nulos — Estoque
df_estoque.isnull().sum()


item                 0
categoria            0
quantidade_atual     0
quantidade_minima    0
ultima_reposicao     0
dtype: int64

In [140]:
# Nulos — Financeiro
df_financeiro.isnull().sum()

data         0
tipo         0
categoria    0
valor        5
descricao    0
dtype: int64

In [141]:
# Duplicatas
df_agenda.duplicated().sum()

np.int64(5)

In [142]:
# Duplicatas
df_estoque.duplicated().sum()

np.int64(0)

In [143]:
# Duplicatas
df_financeiro.duplicated().sum()

np.int64(0)

## Limpeza

### Agenda

In [144]:
# Criar uma cópia do dataframe original pra não modificar.
df_agenda_limpa = df_agenda.copy()

In [145]:
# 1. Remover duplicatas
df_agenda_limpa = df_agenda_limpa.drop_duplicates()

In [146]:
# 2. Padronizar status — tudo minúsculo
df_agenda_limpa['status'] = df_agenda_limpa['status'].str.lower().str.strip()

In [147]:
# 4. Converter data — datas mal formatadas viram NaT e são removidas
df_agenda_limpa['data'] = pd.to_datetime(df_agenda_limpa['data'], errors='coerce')
datas_invalidas = df_agenda_limpa['data'].isna().sum()
df_agenda_limpa = df_agenda_limpa.dropna(subset=['data'])
print(f"Datas inválidas removidas: {datas_invalidas}")

Datas inválidas removidas: 4


In [148]:
df_agenda_limpa.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1213 entries, 0 to 1221
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   data             1213 non-null   datetime64[ns]
 1   horario          1213 non-null   object        
 2   cliente_id       1213 non-null   object        
 3   profissional     1213 non-null   object        
 4   procedimento     1213 non-null   object        
 5   forma_pagamento  1213 non-null   object        
 6   status           1213 non-null   object        
dtypes: datetime64[ns](1), object(6)
memory usage: 75.8+ KB


### Estoque

In [149]:
df_estoque_limpa = df_estoque.copy()

In [150]:
# 1. Corrigir quantidades negativas → 0
# Quantidade negativa não existe fisicamente
df_estoque_limpa['quantidade_atual'] = pd.to_numeric(
    df_estoque_limpa['quantidade_atual'], errors='coerce').clip(lower=0).fillna(0)
df_estoque_limpa['quantidade_minima'] = pd.to_numeric(
    df_estoque_limpa['quantidade_minima'], errors='coerce').clip(lower=0).fillna(0)

In [151]:
# 3. Converter data de reposição
df_estoque_limpa['ultima_reposicao'] = pd.to_datetime(
    df_estoque_limpa['ultima_reposicao'], errors='coerce')

In [152]:
df_estoque_limpa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   item               15 non-null     object        
 1   categoria          15 non-null     object        
 2   quantidade_atual   15 non-null     int64         
 3   quantidade_minima  15 non-null     int64         
 4   ultima_reposicao   15 non-null     datetime64[ns]
dtypes: datetime64[ns](1), int64(2), object(2)
memory usage: 732.0+ bytes


### Financeiro

In [153]:
df_financeiro_limpa = df_financeiro.copy()

In [154]:
# 1. Remover o símbolo R$
df_financeiro_limpa['valor'] = df_financeiro_limpa['valor'].astype(str).str.replace('R$', '', regex=False)

In [155]:
# 2. Remover espaços extras
df_financeiro_limpa['valor'] = df_financeiro_limpa['valor'].str.strip()

In [156]:
# 4. Trocar vírgula decimal por ponto (1500,00 → 1500.00)
df_financeiro_limpa['valor'] = df_financeiro_limpa['valor'].str.replace(',', '.', regex=False)

In [157]:
# 5. Converter para número — valores inválidos viram NaN
df_financeiro_limpa['valor'] = pd.to_numeric(df_financeiro_limpa['valor'], errors='coerce')

# Verificar resultado
df_financeiro_limpa['valor'].dtype

dtype('float64')

In [158]:
df_financeiro_limpa['tipo'].unique()

array(['entrada', 'saida'], dtype=object)

In [159]:
# 3. Converter data
df_financeiro_limpa['data'] = pd.to_datetime(df_financeiro_limpa['data'], errors='coerce')
datas_invalidas = df_financeiro_limpa['data'].isna().sum()
df_financeiro_limpa = df_financeiro_limpa.dropna(subset=['data'])
print(f"Datas inválidas removidas: {datas_invalidas}")

Datas inválidas removidas: 4


In [160]:
# 4. Remover valores negativos e nulos
df_financeiro_limpa = df_financeiro_limpa[df_financeiro_limpa['valor'] > 0]
df_financeiro_limpa = df_financeiro_limpa.dropna(subset=['valor'])

In [161]:
# 5. Adicionar coluna de mês
df_financeiro_limpa['mes'] = df_financeiro_limpa['data'].dt.to_period('M')

> **PROMPT**

> Cria um relatório resumido comparando os DataFrames antes e depois da limpeza (Agenda, Estoque e Financeiro). Pra cada um, mostra quantas linhas tinha originalmente, quantas ficaram depois da limpeza, quantas foram removidas e o percentual de remoção. Formata como uma tabela simples de texto, com título e linhas separadoras pra ficar fácil de ler no console

In [162]:
print("=" * 52)
print("RELATÓRIO DE LIMPEZA")
print("=" * 52)
for nome, original, limpo in [
    ('Agenda',     df_agenda,     df_agenda_limpa),
    ('Estoque',    df_estoque,    df_estoque_limpa),
    ('Financeiro', df_financeiro, df_financeiro_limpa),
]:
    removidos = len(original) - len(limpo)
    pct = removidos / len(original) * 100
    print(f"{nome:<12} {len(original):>5} → {len(limpo):>5}  ({removidos} removidos, {pct:.1f}%)")
print("=" * 52)

RELATÓRIO DE LIMPEZA
Agenda        1222 →  1213  (9 removidos, 0.7%)
Estoque         15 →    15  (0 removidos, 0.0%)
Financeiro     758 →   744  (14 removidos, 1.8%)


## Subir dados limpos pro Drive

In [163]:
salvar_planilha(df_agenda_limpa, "agenda_limpa")
salvar_planilha(df_estoque_limpa, "estoque_limpo")
salvar_planilha(df_financeiro_limpa, "financeiro_limpo")

✓ agenda_limpa
✓ estoque_limpo
✓ financeiro_limpo


## Análise Exploratória

### Taxa de cancelamento por procedimento

> **PROMPT**

> No DataFrame df_agenda_limpa, calcula a porcentagem de agendamentos cancelados pra cada procedimento (quantos cancelados dividido pelo total daquele procedimento). Gera um gráfico de barras com Plotly mostrando essa taxa de cancelamento por procedimento, com título e rótulos de eixo claros.

In [164]:
taxa_cancelamento_proc = df_agenda_limpa.groupby('procedimento')['status'].apply(lambda x: (x == 'cancelado').mean() * 100)
fig = px.bar(x=taxa_cancelamento_proc.index, y=taxa_cancelamento_proc.values, labels={'x':'Procedimento', 'y':'% Cancelado'}, title='Taxa de cancelamento por procedimento (%)')
fig.show()

Olhando o cancelamento por tipo de procedimento, dois pontos se destacam:

- Botox é o procedimento mais confiável da agenda — cancela pouco (12%), o que sugere que é onde o cliente já está mais fidelizado e comprometido com o horário.

- Avaliação inicial, Peeling químico, Microagulhamento e Preenchimento labial concentram os maiores índices de cancelamento (24-25%) — um em cada quatro agendamentos nesses procedimentos não acontece como planejado. Isso representa hora ociosa de profissional e cadeira parada, que poderia estar sendo usada por outro cliente.

- Recomendação: priorizar confirmação ativa (ligação ou lembrete) especificamente pra esses quatro procedimentos, em vez de tratar toda a agenda igual. Se reduzirmos o cancelamento neles pela metade, isso libera uma quantidade relevante de horários por mês sem precisar de nenhum investimento novo — só ajuste de processo.

Ressalva: antes de agir só com base nisso, vale confirmar quantos agendamentos cada procedimento teve no total — se algum desses tiver poucos agendamentos, o percentual pode estar inflado por uma amostra pequena, e a prioridade real pode mudar.


### Procedimentos com mais no-show

> **PROMPT**

>No DataFrame df_agenda_limpa, considera como possível no-show todo agendamento com status 'confirmado' cuja data seja anterior à data mais recente do dataset (ou seja, o dia já passou e o status nunca foi atualizado pra 'realizado' ou 'cancelado'). Agrupa esses casos por procedimento, conta quantos são, ordena do maior pro menor, e gera um gráfico de barras com Plotly mostrando o total de possíveis no-shows por procedimento.

In [165]:
# No-show = confirmado em data passada
# confirmado em data passada = não apareceu e ninguém atualizou o sistema
data_max = df_agenda_limpa['data'].max()
no_show = df_agenda_limpa[
    (df_agenda_limpa['status'] == 'confirmado') &
    (df_agenda_limpa['data'] < data_max)
]

# Contar no-show por procedimento
por_procedimento = no_show.groupby('procedimento').size().reset_index(name='no_show')
por_procedimento = por_procedimento.sort_values('no_show', ascending=False)

print(por_procedimento)

px.bar(por_procedimento, x='procedimento', y='no_show', title='No-show por procedimento').show()

           procedimento  no_show
0     Avaliação inicial       35
3   Harmonização facial       33
7  Preenchimento facial       30
2     Depilação a laser       29
1                 Botox       27
4       Limpeza de pele       24
5      Microagulhamento       24
8  Preenchimento labial       22
6       Peeling químico       21


Avaliação inicial (35) e Harmonização facial (33) lideram em possíveis no-shows — reforçando o padrão já visto no cancelamento, especialmente pra Avaliação inicial.

Mas o dado mais importante aqui é o contraste com Botox: ele tinha a menor taxa de cancelamento (12%), mas aparece com 27 possíveis no-shows — número relativamente alto, próximo do topo da lista. Isso sugere que Botox não tem problema de cliente desistir avisando, mas pode ter problema de cliente simplesmente não aparecer sem avisar. São causas (e soluções) diferentes, mesmo o procedimento parecendo "seguro" quando você olha só cancelamento.

Ressalva rápida: esse número é contagem bruta, não percentual — vale cruzar com o volume total de cada procedimento (do jeito que fizemos antes) antes de comparar diretamente entre eles, porque procedimento com mais agendamentos naturalmente acumula mais casos em número absoluto.

### Receita por procedimento

> **PROMPT**

> No DataFrame df_financeiro_limpa, filtra só as entradas, agrupa por categoria de procedimento e soma o valor recebido em cada uma. Ordena do menor pro maior. Gera um gráfico de barras horizontais com Plotly mostrando a receita total por procedimento

In [166]:
receita = (df_financeiro_limpa[df_financeiro_limpa['tipo'] == 'entrada']
           .groupby('categoria')['valor'].sum()
           .reset_index()
           .sort_values('valor'))

px.bar(receita, x='valor', y='categoria', orientation='h', title='Receita por procedimento').show()

Pacote de tratamento é o maior gerador de receita (~R$109 mil), mas a diferença pros demais procedimentos é pequena — a receita está bem distribuída, sem dependência excessiva de um único serviço. Depilação a laser, Microagulhamento, Preenchimento e Avaliação seguem próximos, todos na faixa de R$85-93 mil.

### Itens críticos de estoque

> **PROMPT**

> No DataFrame df_estoque_limpa, filtra os itens onde a quantidade atual está abaixo da quantidade mínima (itens críticos), e ordena do mais crítico pro menos crítico. Gera um gráfico de barras horizontais com Plotly comparando, pra cada item, a quantidade mínima e a quantidade atual sobrepostas — pra visualizar de forma clara o quanto cada item está abaixo do esperado.

In [167]:
# Filtra itens críticos: quantidade_atual < quantidade_minima
itens_criticos = df_estoque_limpa[df_estoque_limpa['quantidade_atual'] < df_estoque_limpa['quantidade_minima']].copy()
# Ordena do mais crítico para o menos crítico (menor diferença: atual - mínimo)
itens_criticos['deficit'] = itens_criticos['quantidade_atual'] - itens_criticos['quantidade_minima']
itens_criticos = itens_criticos.sort_values('deficit')

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Bar(
    y=itens_criticos['item'],
    x=itens_criticos['quantidade_minima'],
    name='Mínima esperada',
    orientation='h',
    marker_color='rgba(220, 20, 60, 0.4)'
))

fig.add_trace(go.Bar(
    y=itens_criticos['item'],
    x=itens_criticos['quantidade_atual'],
    name='Quantidade atual',
    orientation='h',
    marker_color='rgba(30, 144, 255,0.9)'
))

fig.update_layout(
    barmode='overlay',
    title='Itens críticos do estoque: quantidade atual vs mínima esperada',
    xaxis_title='Quantidade',
    yaxis_title='Item',
    legend=dict(title='Legenda')
)
fig.show()

Toxina botulínica é o item mais crítico — não aparece quantidade atual no gráfico, sinal de estoque zerado, contra um mínimo esperado de 10 unidades. Isso é especialmente grave porque é insumo direto de Botox, um dos procedimentos mais confiáveis da clínica (menor cancelamento) — sem toxina em estoque, a clínica arrisca ter que cancelar ou adiar exatamente os atendimentos que mais geram receita estável.

Luva descartável e Máscara facial hidratante também estão abaixo do mínimo, mas com folga menor (faltam 4 e 3 unidades, respectivamente) — risco moderado, mas ainda dentro de uma margem administrável no curto prazo.

Gaze estéril e Microagulha estão mais próximas do mínimo, risco mais baixo no momento.

Recomendação: reposição urgente de Toxina botulínica antes de qualquer outro item — é o único caso de ruptura total, e o que tem maior impacto direto na receita se não for resolvido rápido.

### Faturamento mensal - entradas vs saidas

> **PROMPT**

> Agrupa df_financeiro_limpa por mês e tipo (entrada/saída), somando os valores. Converte o mês pra texto pra usar como eixo categórico. Gera um gráfico de barras agrupadas com Plotly comparando entrada vs saída mês a mês, com título indicando que os valores estão em reais.

In [168]:
faturamento = df_financeiro_limpa.groupby(['mes', 'tipo'])['valor'].sum().reset_index()
faturamento['mes'] = faturamento['mes'].astype(str)

px.bar(faturamento, x='mes', y='valor', color='tipo', barmode='group',
       title='Entradas vs Saídas por mês (R$)').show()

A clínica opera com margem positiva em todos os meses completos do período — entrada supera saída sem exceção, o que indica saúde financeira consistente. Março e Maio são os meses de maior entrada (~140k e ~135k), enquanto Abril e Junho têm a maior folga proporcional entre entrada e saída (saída bem mais baixa que nos outros meses).

Julho aparece quase zerado, é mês incompleto no dataset (os dados vão só até meados de julho). Vale excluir esse mês da leitura ou marcar visualmente como parcial antes de apresentar, senão passa a falsa impressão de colapso repentino do faturamento.

### Consultas por profissional

> **PROMPT**

>No DataFrame df_agenda_limpa, filtra só os registros com status 'realizado', agrupa por profissional e conta quantos atendimentos cada um teve. Gera um gráfico de barras com Plotly mostrando esse total por profissional.

In [169]:
realizados = df_agenda_limpa[df_agenda_limpa['status'] == 'realizado']
por_profissional = realizados.groupby('profissional').size().reset_index(name='total')

px.bar(por_profissional, x='profissional', y='total', title='Consultas por profissional').show()

## Tendência faturamento

Dentro dessa análise, também dá pra fazer o que se chama de análise de série temporal — que é simplesmente o processo de entender, modelar e prever dados que mudam ao longo do tempo. No nosso caso, o exemplo é o faturamento da clínica: um número que sobe e desce dia após dia, e a gente quer entender esse comportamento pra conseguir dizer algo sobre o que vem a seguir.

Toda série assim carrega três coisas misturadas: uma tendência (o negócio está, no geral, crescendo, caindo ou estável?), uma sazonalidade (existe algum padrão que se repete, tipo mais movimento em certos dias ou meses?) e um ruído (a variação do dia a dia que não segue lógica nenhuma, é só o mundo real sendo imprevisível).

Existem modelos bem sofisticados pra fazer esse tipo de previsão — vocês talvez já tenham ouvido falar de ARIMA, por exemplo. Mas antes de ir pra esse tipo de coisa mais complexa, a prática recomendada é sempre testar primeiro uma abordagem simples, chamada de modelo de linha de base. 

Um deles é a `média móvel` — que é basicamente pegar a média dos últimos n dias (ou meses) e usar isso como sua melhor estimativa pro próximo período. 

Por que começar por algo tão simples, em vez de já ir direto pro modelo mais avançado? Pensa num lojista que quer prever as vendas do mês que vem. Se ele simplesmente calcular a média das vendas dos últimos meses e usar isso como estimativa, essa abordagem básica já costuma acertar cerca de 80% — o suficiente pra ele planejar estoque e organizar a operação sem grande sofisticação nenhuma.

E como a gente sabe se esse modelo simples é confiável, se ele está tentando prever algo que ainda não aconteceu? A resposta é: a gente testa ele contra o que já aconteceu. Pega uma parte recente do histórico que já temos, esconde ela do modelo, gera a previsão só com os dados anteriores, e depois compara com o valor real que já sabemos. Se o modelo chegar perto do que realmente aconteceu, ganhamos confiança pra usar ele de verdade pra prever os próximos dias — sem essa validação, seria só um chute.

---


Média móvel é simplesmente a média dos últimos n períodos — em vez de olhar pro faturamento de uma semana isolada (que ainda pode variar, semana boa, semana fraca), a gente olha pra uma janela de semanas e tira a média. Isso suaviza qualquer oscilação pontual e deixa mais claro pra onde o negócio está indo de verdade.

Vamos calcular duas versões dessa média, com janelas diferentes:

MM3 — média das últimas 3 semanas. Cobre um período curto, então reage rápido — se o faturamento mudar de patamar nas últimas semanas, a MM3 sente isso quase na hora.
MM8 — média das últimas 8 semanas. Cobre um período bem mais amplo, então suaviza ainda mais. Ela muda devagar, e por isso representa o que a gente chama de 'patamar estrutural' do negócio — não é afetada por uma semana isolada boa ou ruim, só se move quando existe uma mudança de verdade que se sustenta ao longo do tempo.

A comparação entre as duas é o que dá o sinal de tendência:

Se a MM3 estiver acima da MM8, significa que o negócio está indo melhor agora do que vinha indo em média — sinal de crescimento.
Se a MM3 estiver abaixo da MM8, significa que o momento atual está pior que a média recente — sinal de desaceleração.

> ANALOGIA

>Imagina uma sala de reunião. Na MM3, você tem 3 pessoas votando — se uma delas muda de opinião, isso já pesa um terço da decisão do grupo. Na MM8, você tem 8 pessoas — se uma muda de opinião, é só 1 voto entre 8, o grupo como um todo quase nem sente.


In [175]:
# Filtra só entradas, garante que 'data' está em formato datetime
entradas = df_financeiro_limpa[df_financeiro_limpa['tipo'] == 'entrada'].copy()
entradas['data'] = pd.to_datetime(entradas['data'])

# Agrupa por dia primeiro (soma tudo que entrou no mesmo dia)
fat_por_dia = entradas.groupby('data')['valor'].sum()

# Agora sim, agrupa por semana (min_count=1 evita semana vazia virar 0 em vez de NaN)
fat_semanal = fat_por_dia.resample('W').sum(min_count=1).to_frame()

fat_semanal.head(10)

,valor
data,
2026-01-04,12372.31
2026-01-11,30597.76
2026-01-18,31325.44
2026-01-25,26376.49
2026-02-01,31051.59
2026-02-08,27147.13
2026-02-15,37577.37
2026-02-22,29594.74
2026-03-01,30757.74


In [176]:
# Pega as últimas semanas pra descrever o "padrão atual"
ultimas_semanas = fat_semanal['valor'].tail(8)  # últimas 8 semanas, por exemplo

ultimas_semanas

data
2026-05-24    33704.20
2026-05-31    34716.97
2026-06-07    31928.94
2026-06-14    40060.50
2026-06-21    42249.11
2026-06-28     9432.85
2026-07-05     5011.00
2026-07-12     2872.26
Freq: W-SUN, Name: valor, dtype: float64

In [179]:
media = ultimas_semanas.mean()
minimo = ultimas_semanas.min()
maximo = ultimas_semanas.max()

In [186]:
# 2. Calcula as duas médias móveis — série completa, não só o último ponto
fat_semanal['mm3'] = fat_semanal['valor'].rolling(window=3).mean()
fat_semanal['mm8'] = fat_semanal['valor'].rolling(window=8).mean()

In [187]:
# 3. Gráfico: faturamento real + as duas médias móveis sobrepostas
fig = go.Figure()

fig.add_trace(go.Bar(
    x=fat_semanal.index, y=fat_semanal['valor'],
    name='Faturamento semanal', marker_color='lightsteelblue', opacity=0.5
))
fig.add_trace(go.Scatter(
    x=fat_semanal.index, y=fat_semanal['mm3'],
    name='MM3 (curto prazo)', line=dict(color='steelblue', width=2)
))
fig.add_trace(go.Scatter(
    x=fat_semanal.index, y=fat_semanal['mm8'],
    name='MM8 (base estrutural)', line=dict(color='tomato', width=2.5)
))

fig.update_layout(
    title='Tendência de faturamento semanal — MM3 vs MM8',
    yaxis_title='R$',
    hovermode='x unified'
)
fig.show()

A comparação existe pra transformar dois números soltos (mm_curta e mm_longa) numa resposta de negócio direta: "estamos indo melhor, pior, ou igual à nossa própria média recente?" — é o mesmo princípio do cruzamento MM7 x MM30 que vocês já usaram no diário, só que aplicado aqui como teste único (não como gráfico contínuo).

A lógica por trás:

mm_curta representa "como estivemos agora, recentemente" (últimas 3 semanas)
mm_longa representa "como estivemos numa base mais ampla" (últimas 8 semanas)

Se mm_curta está acima de mm_longa, isso quer dizer: o momento mais recente está puxando a média pra cima — sinal de que o negócio está performando melhor agora do que vinha performando em média. Se está abaixo, o oposto: o momento recente está pior que a média histórica próxima — sinal de desaceleração.

Por que comparar duas médias, e não só olhar um valor absoluto?
Porque "R$30 mil essa semana" sozinho não diz nada — é bom ou ruim comparado a quê? Comparando com a própria média recente do negócio, você cria uma referência interna, sem precisar de benchmark externo (mercado, concorrente). É "estamos melhores ou piores que nós mesmos, ultimamente?"

E por que a margem de 2% (* 1.02 / * 0.98) em vez de comparar direto mm_curta > mm_longa?
Sem essa margem, qualquer diferença mínima (tipo R$50 de diferença entre as duas médias) já dispararia "tendência de alta" ou "queda" — o que soaria como ruído sendo tratado como sinal real. A margem de 2% exige que a diferença seja grande o suficiente pra significar algo, evitando alarme falso por flutuação pequena e natural.

Conectando com o que acabamos de descobrir: é exatamente essa comparação que, rodada nos dados de vocês agora, vai gritar "queda" — porque mm_curta (5.772, distorcida pelo buraco de dado) está muito abaixo de mm_longa (24.996, mais estável). A comparação está funcionando certinho tecnicamente — o problema não é a lógica da comparação, é a qualidade do dado que ela está recebendo, que é exatamente a lição que a gente vem reforçando: o código pode estar perfeito e ainda assim entregar conclusão errada, se o dado de entrada estiver com problema.




In [189]:
# Pega só o valor mais recente de cada média móvel (o "agora")
mm_curta_atual = fat_semanal['mm3'].iloc[-1]
mm_longa_atual = fat_semanal['mm8'].iloc[-1]


# Compara direto — mesmo princípio do "Golden Cross" / "Death Cross" do mercado financeiro
if mm_curta_atual > mm_longa_atual:
    tendencia = "tendência de alta"
elif mm_curta_atual < mm_longa_atual:
    tendencia = "tendência de queda"
else:
    tendencia = "estável, sem tendência clara"

print(f"MM3 atual: R$ {mm_curta_atual:,.2f} | MM8 atual: R$ {mm_longa_atual:,.2f}")
print(f"Leitura: {tendencia}")

MM3 atual: R$ 5,772.04 | MM8 atual: R$ 24,996.98
Leitura: tendência de queda


## TAREFA AULA 1

- Criar funções para os KPIs
    - Taxa de cancelamento
    - Estoque crítico
    - Taxa de ocupação